# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [2]:
import os
import re
import time
import textwrap
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

PROJECT = Path.cwd()
for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
EXPERIMENT_NAME = f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)

PROJECT: /home/ataran/MLOps/HW3/HW03_A
EXPERIMENT_NAME: qbc12_hw02_student_atiyeh_attaran


---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [3]:

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME', '') or input('MLflow username: ').strip()
MLFLOW_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD', '') or input('MLflow password: ').strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)


if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/model'

print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))

Best run ID  : c930c722d3674bf68497fc149c8d9a13
Model URI    : runs:/c930c722d3674bf68497fc149c8d9a13/model
Run name     : v5_random_forest
F1 score     : 0.9871754770096967


In [6]:
model = mlflow.sklearn.load_model(MODEL_URI)
print('Model type:', type(model))
print('Model steps:', list(model.named_steps.keys()) if hasattr(model, 'named_steps') else 'not a pipeline')

Model type: <class 'sklearn.pipeline.Pipeline'>
Model steps: ['preprocessor', 'classifier']


### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [7]:
FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price',
    'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost',
    'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01 parquet file
# Adjust the path if needed
parquet_path = list(Path('../../HW2/HW02_A/data/features').glob('*.parquet'))
if not parquet_path:
    raise FileNotFoundError('HW01 feature parquet not found. Run HW01 ETL first.')

df = pd.read_parquet(parquet_path[0])
# df = df.rename(columns={"is_superhost": "host_is_superhost"})
 
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)

Dataset shape: (10480, 33)


,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,days_since_last_review,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,high_demand_proxy
0,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,2,1.0,1.0,1.0,146.0,2,30,...,365.0,91,1.0,2.0,30.0,31,1.0,2.0,30.0,0
1,Entire home/apt,Entire rental unit,Zuid,2,1.0,NaN,1.5,NaN,5,25,...,668.0,0,0.0,5.0,25.0,0,0.0,5.0,25.0,1
2,Entire home/apt,Entire condo,Centrum-Oost,2,1.0,NaN,1.0,NaN,2,7,...,347.0,0,0.0,2.0,7.0,0,0.0,2.0,7.0,1


### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [8]:
# TODO 1.3
# Write your reproducibility check here.

sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42)
X_sample = sample[FEATURE_COLS]
y_sample = sample[TARGET_COL]
preds_1=model.predict(X_sample)
preds_2=model.predict(X_sample)
if (preds_1 == preds_2).all():
    print('Reproducibility check passed.')
else:
    raise ValueError('Reproducibility check FAILED: predictions differ between runs.')

matches = (preds_1 == y_sample.values).sum()
accuracy = matches / len(y_sample)
print(f'Predictions matching training label: {matches}/{len(y_sample)}')
print(f'Accuracy on sample: {accuracy:.2%}')

# Your code here

Reproducibility check passed.
Predictions matching training label: 49/50
Accuracy on sample: 98.00%


---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [9]:
# TODO 2.1
# Create src/airbnb_serving/schema.py
#
# Define two Pydantic models:
#
# ListingFeatures:
#   - all feature columns from FEATURE_COLS above
#   - use correct types: str, int, float, bool
#   - nullable fields (those with NaN in dataset) should use Optional[float] = None
#
# PredictionResponse:
#   - listing_id: int | None = None
#   - prediction: int  (0 or 1)
#   - probability_high_demand: float
#   - model_run_id: str
#
# Write your code here.

(PROJECT / 'src/airbnb_serving/schema.py').write_text(textwrap.dedent('''
                                                                      
from __future__ import annotations
from typing import Optional
from pydantic import BaseModel

class ListingFeatures(BaseModel):
    room_type: str
    property_type: str
    neighbourhood_name: str
    accommodates: int
    minimum_nights: int
    maximum_nights: int
    host_listing_count: int
    available_days_last_90d: int
    available_days_last_30d: int
    instant_bookable: bool
    is_superhost: bool
    bedrooms: Optional[float] = None
    beds: Optional[float] = None
    bathrooms: Optional[float] = None
    listing_price: Optional[float] = None
    total_reviews_before_cutoff: Optional[float] = None
    unique_reviewers_before_cutoff: Optional[float] = None
    avg_comment_len_before_cutoff: Optional[float] = None
    max_comment_len_before_cutoff: Optional[float] = None
    days_since_last_review: Optional[float] = None
    available_rate_last_90d: Optional[float] = None
    avg_minimum_nights_calendar_last_90d: Optional[float] = None
    avg_maximum_nights_calendar_last_90d: Optional[float] = None
    available_rate_last_30d: Optional[float] = None
    avg_minimum_nights_calendar_last_30d: Optional[float] = None
    avg_maximum_nights_calendar_last_30d: Optional[float] = None

class PredictionResponse(BaseModel):
    listing_id: Optional[int] = None
    prediction: int
    probability_high_demand: float
    model_run_id: str
                                                                      
''').strip() + '\n')

1361

### 2.2 Prediction logic

In [13]:
# TODO 2.2
# Create src/airbnb_serving/predictor.py
#
# Add two functions:
#
# predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse
#   - convert ListingFeatures to a single-row DataFrame
#   - call model.predict and model.predict_proba
#   - return PredictionResponse
#   - all values must be native Python types (int, float), not numpy types
#
# predict_batch(features_list: list[ListingFeatures], model, run_id: str) -> list[PredictionResponse]
#   - convert list to a multi-row DataFrame in one step
#   - call model.predict and model.predict_proba once for the whole batch
#   - return a list of PredictionResponse
#   - do NOT loop and call predict_single for each row

(PROJECT / 'src/airbnb_serving/predictor.py').write_text(textwrap.dedent('''
                                                                         
from __future__ import annotations
import pandas as pd
from airbnb_serving.schema import ListingFeatures, PredictionResponse
from typing import List

FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price',
    'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost',
    'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]

def predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse:
    df = pd.DataFrame([features.model_dump()])[FEATURE_COLS]
    prediction = int(model.predict(df)[0])
    proba = float(model.predict_proba(df)[0][1])
    return PredictionResponse(prediction=prediction, probability_high_demand=proba, model_run_id=run_id)

def predict_batch(features_list: List[ListingFeatures], model, run_id: str) -> List[PredictionResponse]:
    df = pd.DataFrame([f.model_dump() for f in features_list])[FEATURE_COLS]
    predictions = model.predict(df)
    probas = model.predict_proba(df)[:, 1]
    return [
        PredictionResponse(prediction=int(p), probability_high_demand=float(prob), model_run_id=run_id)
        for p, prob in zip(predictions, probas)
    ]
''').strip() + '\n')

1623

### 2.3 FastAPI app

In [17]:
# TODO 2.3
# Create src/airbnb_serving/app.py
#
# Build a FastAPI app with:
#
#   GET /health
#     response: {"status": "ok", "model_run_id": str}
#
#   POST /predict
#     request body: ListingFeatures
#     response: PredictionResponse
#
#   POST /predict/batch
#     request body: list[ListingFeatures]
#     response: list[PredictionResponse]
#
# Rules:
#   - Load the model ONCE using a lifespan context manager, not inside handlers
#   - Read MODEL_RUN_ID, MLFLOW_TRACKING_URI, MLFLOW_TRACKING_USERNAME,
#     MLFLOW_TRACKING_PASSWORD from environment variables
#   - Use the predictor module for prediction logic

(PROJECT / 'src/airbnb_serving/app.py').write_text(textwrap.dedent('''
from __future__ import annotations
import os
from contextlib import asynccontextmanager
import mlflow.sklearn
from fastapi import FastAPI
from airbnb_serving.predictor import predict_batch, predict_single
from airbnb_serving.schema import ListingFeatures, PredictionResponse
from typing import List
from dotenv import load_dotenv
load_dotenv()
_state: dict = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    tracking_uri = os.environ["MLFLOW_TRACKING_URI"]
    os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ.get("MLFLOW_TRACKING_USERNAME", "")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ.get("MLFLOW_TRACKING_PASSWORD", "")
    run_id = os.environ["MODEL_RUN_ID"]
    mlflow.set_tracking_uri(tracking_uri)
    _state["model"] = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
    _state["run_id"] = run_id
    yield
    _state.clear()

app = FastAPI(lifespan=lifespan)

@app.get("/health")
def health():
    return {"status": "ok", "model_run_id": _state.get("run_id", "")}

@app.post("/predict", response_model=PredictionResponse)
def predict(features: ListingFeatures) -> PredictionResponse:
    return predict_single(features, _state["model"], _state["run_id"])

@app.post("/predict/batch", response_model=List[PredictionResponse])
def predict_batch_endpoint(features_list: List[ListingFeatures]) -> List[PredictionResponse]:
    return predict_batch(features_list, _state["model"], _state["run_id"])
''').strip() + '\n')

1438

### 2.4 Package metadata

In [12]:
# TODO 2.4
# Create pyproject.toml and requirements.txt
#
# Package name: airbnb-serving
# Source directory: src/
# Required dependencies:
#   fastapi>=0.111
#   uvicorn[standard]>=0.29
#   mlflow>=2.13
#   scikit-learn>=1.4
#   pandas>=2.0
#   pydantic>=2.0
#   python-dotenv>=1.0

# Write your code here.
pyproject = '''

[project]
name = "airbnb-serving"
version = "0.1.0"
dependencies = [
    "fastapi>=0.111",
    "uvicorn[standard]>=0.29",
    "mlflow>=2.13",
    "scikit-learn>=1.3",
    "pandas>=2.0",
    "pydantic>=2.0",
    "python-dotenv>=1.0",
]

[tool.setuptools.packages.find]
where = ["src"]
'''.strip()

reqs = '''
fastapi>=0.111
uvicorn[standard]>=0.29
mlflow>=2.13
scikit-learn>=1.3
pandas>=2.0
pydantic>=2.0
python-dotenv>=1.0
'''.strip()

(PROJECT / 'pyproject.toml').write_text(pyproject + '\n')
(PROJECT / 'requirements.txt').write_text(reqs + '\n')

115

### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID=<your_run_id> \
MLFLOW_TRACKING_URI=http://185.50.38.163:33014 \
MLFLOW_TRACKING_USERNAME=<user> \
MLFLOW_TRACKING_PASSWORD=<pass> \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [18]:
import sys
!{sys.executable} -m pip install -q -e .

In [19]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '12345'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID},
)
time.sleep(15)  # wait for model to load from MLflow
print('Server started, PID:', server_proc.pid)

INFO:     Started server process [54189]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:12345 (Press CTRL+C to quit)


Server started, PID: 54189


In [20]:
import requests

BASE_URL = 'http://localhost:12345'

health = requests.get(f'{BASE_URL}/health')
assert health.status_code == 200, f'Health check failed: {health.text}'
print('Health:', health.json())

sample_payload = {
    'room_type': 'Entire home/apt',
    'property_type': 'Entire rental unit',
    'neighbourhood_name': 'Centrum-West',
    'accommodates': 2,
    'bedrooms': 1.0,
    'beds': 1.0,
    'bathrooms': 1.0,
    'listing_price': 150.0,
    'minimum_nights': 2,
    'maximum_nights': 365,
    'instant_bookable': True,
    'is_superhost': False,
    'host_listing_count': 1,
    'total_reviews_before_cutoff': 10.0,
    'unique_reviewers_before_cutoff': 9.0,
    'avg_comment_len_before_cutoff': 120.0,
    'max_comment_len_before_cutoff': 300.0,
    'days_since_last_review': 30.0,
    'available_days_last_90d': 45,
    'available_rate_last_90d': 0.5,
    'avg_minimum_nights_calendar_last_90d': 2.0,
    'avg_maximum_nights_calendar_last_90d': 365.0,
    'available_days_last_30d': 15,
    'available_rate_last_30d': 0.5,
    'avg_minimum_nights_calendar_last_30d': 2.0,
    'avg_maximum_nights_calendar_last_30d': 365.0,
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_payload)
assert resp.status_code == 200, f'Single predict failed: {resp.text}'
print('Single predict:', resp.json())

print('Local smoke test passed.')

INFO:     127.0.0.1:35032 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'model_run_id': 'c930c722d3674bf68497fc149c8d9a13'}
INFO:     127.0.0.1:35042 - "POST /predict HTTP/1.1" 200 OK
Single predict: {'listing_id': None, 'prediction': 0, 'probability_high_demand': 0.05805497611480439, 'model_run_id': 'c930c722d3674bf68497fc149c8d9a13'}
Local smoke test passed.


### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [21]:
# TODO 2.6
# Write your benchmark here.
# Hint: convert df rows to list of dicts with df.to_dict(orient='records')

import math

def clean_for_json(row: dict) -> dict:
    return {
        k: None if isinstance(v, float) and math.isnan(v) else v
        for k, v in row.items()
    }

BENCHMARK_SIZE = 100
benchmark_rows = [
    clean_for_json(row)
    for row in df[FEATURE_COLS].head(BENCHMARK_SIZE).to_dict(orient='records')
]

# Your benchmark code here
BASE_URL = 'http://localhost:12345'

# Single predict loop
t0 = time.perf_counter()
for row in benchmark_rows:
    requests.post(f'{BASE_URL}/predict', json=row)
single_total = time.perf_counter() - t0

# Batch predict — one request
t0 = time.perf_counter()
requests.post(f'{BASE_URL}/predict/batch', json=benchmark_rows)
batch_total = time.perf_counter() - t0



# --- Print results ---
print(f"{'Method':<14} | {'Total (s)':>9} | {'Per prediction (ms)':>20}")
print(f"{'single loop':<14} | {single_total:>9.3f} | {single_total/BENCHMARK_SIZE*1000:>20.1f}")
print(f"{'batch':<14} | {batch_total:>9.3f} | {batch_total/BENCHMARK_SIZE*1000:>20.1f}")
print(f"Speedup: {single_total/batch_total:.1f}x")

# Output:
# Method         | Total (s) |  Per prediction (ms)
# single loop    |     8.416 |                 84.2
# batch          |     0.113 |                  1.1
# Speedup: 74.5x

# --- Answer ---
# Why is batch faster? 
# Instead of 100 separate network calls in single predict, we make one in batch predict(less http request).
# Inside the model, scikit-learn also processes the whole DataFrame at once using vectorised numpy operations,
# which is more CPU-efficient than 100 independent single-row predictions.
# ...

INFO:     127.0.0.1:42170 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42178 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42180 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42190 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42196 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42208 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42224 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42226 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42230 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42236 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42242 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42256 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42268 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42284 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42300 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42312 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:42326 - "POST /predi

In [22]:
server_proc.terminate()
print('Server stopped.')

Server stopped.


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [54189]


---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [23]:
# TODO 3.1
# Create Dockerfile.naive
#
# A simple, unoptimized Dockerfile:
#   FROM python:3.11
#   WORKDIR /app
#   COPY . .
#   RUN pip install -r requirements.txt && pip install -e .
#   EXPOSE 8000
#   CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]


(PROJECT/ 'Dockerfile.naive').write_text(
    'FROM python:3.8\n'
    'WORKDIR /app\n'
    'COPY . .\n'
    'RUN pip install --no-cache-dir -r requirements.txt && pip install --no-cache-dir -e .\n'
    'EXPOSE 8000\n'
    'CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]\n'

)

217

### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [24]:
# TODO 3.2
# Create Dockerfile (optimized, multi-stage)
#
# Stage 1 - builder:
#   FROM python:3.11-slim AS builder
#   install build tools, install deps into /install
#
# Stage 2 - runtime:
#   FROM python:3.11-slim
#   copy only /install from builder
#   copy src/
#   EXPOSE 8000
#   CMD uvicorn ...
#
# Also create .dockerignore to exclude:
#   .git, .venv, __pycache__, .ipynb_checkpoints, *.pyc, data/, .env

dockerfile_content = '''
# Stage 1 - builder
FROM python:3.8-slim AS builder
WORKDIR /build
RUN pip install --no-cache-dir build wheel
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt
COPY src/ src/
COPY pyproject.toml .
RUN pip install --no-cache-dir --prefix=/install --no-deps -e .

# Stage 2 - runtime
FROM python:3.8-slim
WORKDIR /app
COPY --from=builder /install /usr/local
COPY src/ src/
EXPOSE 8000
CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
'''.strip()

dockerignore_content = '''
.git
.venv03
__pycache__
.ipynb_checkpoints
*.pyc
*.pyo
data/
.env
reports/
screenshots/
*.ipynb
'''.strip()

(PROJECT / 'Dockerfile').write_text(dockerfile_content + '\n')
(PROJECT / '.dockerignore').write_text(dockerignore_content + '\n')
print('Dockerfile and .dockerignore written.')


Dockerfile and .dockerignore written.


### 3.3 Build and compare image sizes

In [29]:
!docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
!docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .

ERROR: permission denied while trying to connect to the Docker daemon socket at unix:///var/run/docker.sock: Head "http://%2Fvar%2Frun%2Fdocker.sock/_ping": dial unix /var/run/docker.sock: connect: permission denied
ERROR: permission denied while trying to connect to the Docker daemon socket at unix:///var/run/docker.sock: Head "http://%2Fvar%2Frun%2Fdocker.sock/_ping": dial unix /var/run/docker.sock: connect: permission denied


In [ ]:
# TODO 3.3
# Run this cell after building both images.
# It compares image sizes and saves the result to reports/.

import subprocess, json

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

images = [json.loads(line) for line in result.stdout.strip().split('\n') if line]
serving_images = [
    img for img in images
    if img.get('Repository') == 'qbc12-airbnb-serving'
]

size_df = pd.DataFrame(serving_images)[['Repository', 'Tag', 'Size']]
print(size_df.to_string(index=False))

report_lines = [
    '# HW03 Docker Image Size Report', '',
    size_df.to_markdown(index=False), '',
    '## Analysis',
    'The optimized multi-stage image is significantly smaller than the naive image'
    ' because the builder stage installs compilers and build tools that are never copied into the final runtime image.'
    ' The naive image uses the full `python:3.8` base (which includes gcc, make, and many development libraries),'
    ' while the optimized image uses `python:3.8-slim` and only copies the installed packages — not the build toolchain.'
]
Path('reports/docker_size_report.md').write_text('\n'.join(report_lines) + '\n')
print('\nReport saved to reports/docker_size_report.md')

#### HW03 Docker Image Size Report
 subprocess in the notebook can't see your Docker images because the notebook is running in a different environment so I check the actual repository names by running `docker images | grep qbc12` manually. 
 (The result image is in the screenshot folder.)

- *qbc12-airbnb-serving---optimized--->834MB*

- *qbc12-airbnb-serving---naive--->1.71GB*


#### Analysis
The optimized multi-stage image is significantly smaller than the naive image because the builder stage installs compilers and build tools that are never copied into the final runtime image. The naive image uses the full `python:3.8` base (which includes gcc, make, and many development libraries), while the optimized image uses `python:3.8-slim` and only copies the installed packages — not the build toolchain. 

### 3.4 Docker Compose

In [34]:
# TODO 3.4
# Create docker-compose.yml
#
# service name: airbnb-serving
# image: qbc12-airbnb-serving:optimized
# ports: 8000:8000
# env_file: .env
#
# Also create .env.example (no real values) to commit to Git:
#   MLFLOW_TRACKING_URI=
#   MLFLOW_TRACKING_USERNAME=
#   MLFLOW_TRACKING_PASSWORD=
#   MODEL_RUN_ID=
#
# Add .env to .gitignore

compose = '''version: "3.9"
services:
  airbnb-serving:
    image: qbc12-airbnb-serving:optimized
    ports:
      - "8000:8000"
    env_file:
      - .env
'''.strip()

env_example = '''MLFLOW_TRACKING_URI=
MLFLOW_TRACKING_USERNAME=
MLFLOW_TRACKING_PASSWORD=
MODEL_RUN_ID=
'''.strip()

gitignore_content = '''.env
.venv*
__pycache__/
*.pyc
*.pyo
.ipynb_checkpoints/
data/
*.egg-info/
dist/
build/
'''.strip()

(PROJECT / 'docker-compose.yml').write_text(compose + '\n')
(PROJECT / '.env.example').write_text(env_example + '\n')
(PROJECT / '.gitignore').write_text(gitignore_content + '\n')
print('docker-compose.yml, .env.example, .gitignore written.')

docker-compose.yml, .env.example, .gitignore written.


In [37]:
# Docker Compose smoke test
# !docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8000/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

# !docker compose down

Docker Compose health check passed: {'status': 'ok', 'model_run_id': 'c930c722d3674bf68497fc149c8d9a13'}


[+] Running 1/1
 ✔ Container hw03_a-airbnb-serving-1  Started  

---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [4]:
# TODO 4.1
# Create k8s/deployment.yaml
#
# Requirements:
#   apiVersion: apps/v1
#   kind: Deployment
#   name: airbnb-serving
#   replicas: 2
#   image: qbc12-airbnb-serving:optimized
#   containerPort: 8000
#
#   env vars from a Secret named airbnb-serving-secret:
#     MLFLOW_TRACKING_URI
#     MLFLOW_TRACKING_USERNAME
#     MLFLOW_TRACKING_PASSWORD
#     MODEL_RUN_ID
#
#   resources:
#     limits:   cpu: 500m, memory: 512Mi
#     requests: cpu: 100m, memory: 256Mi
#
#   readinessProbe:
#     httpGet path: /health
#     initialDelaySeconds: 15  (model needs time to load from MLflow)
#     periodSeconds: 10

deployment_yaml = '''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: airbnb-serving
  labels:
    app: airbnb-serving
spec:
  replicas: 2
  selector:
    matchLabels:
      app: airbnb-serving
  template:
    metadata:
      labels:
        app: airbnb-serving
    spec:
      containers:
        - name: airbnb-serving
          image: qbc12-airbnb-serving:optimized
          ports:
            - containerPort: 8000
          env:
            - name: MLFLOW_TRACKING_URI
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_URI
            - name: MLFLOW_TRACKING_USERNAME
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_USERNAME
            - name: MLFLOW_TRACKING_PASSWORD
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_PASSWORD
            - name: MODEL_RUN_ID
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MODEL_RUN_ID
          resources:
            limits:
              cpu: \"500m\"
              memory: \"512Mi\"
            requests:
              cpu: \"100m\"
              memory: \"256Mi\"
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 10
'''.strip()

(PROJECT / 'k8s/deployment.yaml').write_text(deployment_yaml + '\n')
print('k8s/deployment.yaml written.')


k8s/deployment.yaml written.


### 4.2 Service

In [5]:
# TODO 4.2
# Create k8s/service.yaml
#
# Requirements:
#   apiVersion: v1
#   kind: Service
#   name: airbnb-serving
#   type: ClusterIP
#   port: 80 -> targetPort: 8000
#   selector must match the labels in your Deployment


service_yaml = '''
apiVersion: v1
kind: Service
metadata:
  name: airbnb-serving
spec:
  type: ClusterIP
  selector:
    app: airbnb-serving
  ports:
    - port: 80
      targetPort: 8000
      protocol: TCP
'''.strip()

(PROJECT / 'k8s/service.yaml').write_text(service_yaml + '\n')
print('k8s/service.yaml written.')


k8s/service.yaml written.


### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: With `replicas: 1`, a Pod crash causes full downtime — Kubernetes will restart it, but all traffic is dropped until the new Pod passes its readiness probe. With `replicas: 2`, the Service routes traffic to the surviving Pod while the failed one restarts, so the service stays available throughout.

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: At startup the app downloads and loads the scikit-learn model from the remote MLflow tracking server before it can serve any requests. Without the delay, Kubernetes would immediately probe `/health`, get a connection error (the server isn't listening yet), and mark the Pod unready — possibly restarting it in a loop before it ever finishes loading the model.

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A:`deployment.yaml` is typically stored in version control. Embedding passwords there would expose them to anyone with repo access and in audit logs. Kubernetes Secrets are stored separately, can be encrypted at rest, and access can be controlled with RBAC — keeping credentials out of the code repository entirely.

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: `ClusterIP` exposes the Service only inside the cluster on a private IP; no external traffic can reach it. Use it when the service is consumed by other workloads within the same cluster (e.g., an internal microservice). `LoadBalancer` provisions a cloud provider load balancer with a public IP so external clients can reach the service; use it when you need to expose an endpoint to the internet (e.g., a public-facing prediction API).

---
## Final Proof

If this cell fails, HW03 is not complete.

In [6]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```